In [1]:
pip install category_encoders

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 1.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import category_encoders as ce
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler


In [3]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [4]:
data=pd.read_csv(r"/content/features_having_most_influence_on_Botnet_IoT.csv")


In [ ]:
'''
data=pd.read_csv(r"/content/drive/MyDrive/iot_botnet_dataset/features_having_most_influence_on_Botnet_IoT.csv")
data=pd.read_csv(r"/content/drive/MyDrive/Group 10 - IBM Project - Sandeep and Sudhay/Temp_Folders/Datasets/features_having_most_influence_on_Botnet_IoT.csv")
'''


'\ndata=pd.read_csv(r"/content/drive/MyDrive/iot_botnet_dataset/features_having_most_influence_on_Botnet_IoT.csv")\ndata=pd.read_csv(r"/content/drive/MyDrive/Group 10 - IBM Project - Sandeep and Sudhay/Temp_Folders/Datasets/features_having_most_influence_on_Botnet_IoT.csv")\n'

In [5]:
data.head()

,pkSeqID,proto,saddr,sport,daddr,dport,seq,stddev,N_IN_Conn_P_SrcIP,min,state_number,mean,N_IN_Conn_P_DstIP,drate,srate,max,attack,category,subcategory
0,792371,udp,192.168.100.150,48516,192.168.100.3,80,175094,0.226784,100,4.100436,4,4.457383,100,0.000000,0.404711,4.719438,1,DoS,UDP
1,2056418,tcp,192.168.100.148,22267,192.168.100.3,80,143024,0.451998,100,3.439257,1,3.806172,100,0.225077,0.401397,4.442930,1,DDoS,TCP
2,2795650,udp,192.168.100.149,28629,192.168.100.3,80,167033,1.931553,73,0.000000,4,2.731204,100,0.000000,0.407287,4.138455,1,DDoS,UDP
3,2118009,tcp,192.168.100.148,42142,192.168.100.3,80,204615,0.428798,56,3.271411,1,3.626428,100,0.000000,0.343654,4.229700,1,DDoS,TCP
4,303688,tcp,192.168.100.149,1645,192.168.100.5,80,40058,2.058381,100,0.000000,3,1.188407,100,0.000000,0.135842,4.753628,1,DoS,TCP


In [6]:
len(data)


733705

In [7]:
data.dtypes

pkSeqID                int64
proto                 object
saddr                 object
sport                 object
daddr                 object
dport                 object
seq                    int64
stddev               float64
N_IN_Conn_P_SrcIP      int64
min                  float64
state_number           int64
mean                 float64
N_IN_Conn_P_DstIP      int64
drate                float64
srate                float64
max                  float64
attack                 int64
category              object
subcategory           object
dtype: object

In [8]:
df = data.drop(['pkSeqID','subcategory','attack','dport','sport'],axis=1)

In [9]:
#encoder1 = ce.HashingEncoder(cols='saddr',n_components=6)

encoder1 = ce.BinaryEncoder(cols=['saddr'],return_df=True)
df = encoder1.fit_transform(df)

In [10]:
encoder2 = ce.BinaryEncoder(cols=['daddr'],return_df=True)
df = encoder2.fit_transform(df)

In [11]:
proto_encoded = pd.get_dummies(data=df['proto'],drop_first=True)

In [12]:
df = pd.concat([df,proto_encoded],axis=1)
df.drop('proto',axis=1,inplace=True)

In [13]:
df.dtypes

saddr_0                int64
saddr_1                int64
saddr_2                int64
saddr_3                int64
saddr_4                int64
daddr_0                int64
daddr_1                int64
daddr_2                int64
daddr_3                int64
daddr_4                int64
daddr_5                int64
seq                    int64
stddev               float64
N_IN_Conn_P_SrcIP      int64
min                  float64
state_number           int64
mean                 float64
N_IN_Conn_P_DstIP      int64
drate                float64
srate                float64
max                  float64
category              object
icmp                   uint8
ipv6-icmp              uint8
tcp                    uint8
udp                    uint8
dtype: object

In [14]:
scaler = StandardScaler()
X = df.drop('category',axis=1)
y = df['category']
X = scaler.fit_transform(X)

In [15]:
X_train,X_test,y_train,y_test = train_test_split(X,y,train_size=0.8,random_state=109)

In [16]:
clf = LogisticRegression(random_state=0,multi_class='multinomial').fit(X_train, y_train)


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [17]:
predictions = clf.predict(X_test)
print(accuracy_score(y_test,predictions))

0.9853347053652354


In [18]:
# Multinomial Logistic Regression
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
print("\t\tClassification report for Multinomial Logistic Regression\n\n",classification_report(y_test,predictions,digits=6))
print()
print()
print("Confusion Matrix for Multinomial Logistic Regression")
print(confusion_matrix(y_test,predictions))


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


		Classification report for Multinomial Logistic Regression

                 precision    recall  f1-score   support

          DDoS   0.994369  0.984061  0.989188     76983
           DoS   0.979881  0.996145  0.987946     66152
        Normal   0.809524  0.944444  0.871795        18
Reconnaissance   0.888584  0.814226  0.849782      3585
         Theft   0.000000  0.000000  0.000000         3

      accuracy                       0.985335    146741
     macro avg   0.734472  0.747775  0.739742    146741
  weighted avg   0.985210  0.985335  0.985188    146741



Confusion Matrix for Multinomial Logistic Regression
[[75756  1065     1   161     0]
 [   50 65897     2   203     0]
 [    0     0    17     1     0]
 [  379   286     1  2919     0]
 [    0     2     0     1     0]]


In [48]:
import keras
from keras.models import Sequential
from keras.layers import Dense

In [49]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()
y = df['category'].values

y = ohe.fit_transform(y.reshape(-1,1)).toarray()

X_train,X_test,y_train,y_test = train_test_split(X,y,train_size=0.8,random_state=109)


In [50]:
model = Sequential()
model.add(Dense(16,input_dim=25,activation='relu'))
model.add(Dense(12,activation='relu'))
model.add(Dense(5,activation='softmax'))

In [51]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [52]:
history = model.fit(X_train, y_train, epochs=100, batch_size=64)

Epoch 1/100
9172/9172 [==============================] - 24s 2ms/step - loss: 0.0688 - accuracy: 0.9780
Epoch 2/100
9172/9172 [==============================] - 25s 3ms/step - loss: 0.0264 - accuracy: 0.9900
Epoch 3/100
9172/9172 [==============================] - 25s 3ms/step - loss: 0.0233 - accuracy: 0.9910
Epoch 4/100
9172/9172 [==============================] - 23s 2ms/step - loss: 0.0214 - accuracy: 0.9916
Epoch 5/100
9172/9172 [==============================] - 21s 2ms/step - loss: 0.0197 - accuracy: 0.9923
Epoch 6/100
9172/9172 [==============================] - 21s 2ms/step - loss: 0.0186 - accuracy: 0.9928
Epoch 7/100
9172/9172 [==============================] - 21s 2ms/step - loss: 0.0176 - accuracy: 0.9934
Epoch 8/100
9172/9172 [==============================] - 23s 2ms/step - loss: 0.0161 - accuracy: 0.9939
Epoch 9/100
9172/9172 [==============================] - 21s 2ms/step - loss: 0.0150 - accuracy: 0.9944
Epoch 10/100
9172/9172 [==============================] - 21s 2m

In [53]:
y_pred = model.predict(X_test)

4586/4586 [==============================] - 9s 2ms/step


In [54]:
pred = list()
for i in range(len(y_pred)):
    pred.append(np.argmax(y_pred[i]))
test = list()
for i in range(len(y_test)):
      test.append(np.argmax(y_test[i]))

In [55]:
from sklearn.metrics import accuracy_score
a = accuracy_score(test,pred)
print('Accuracy is:', a)

Accuracy is: 0.9972536646199767


In [56]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
print("\t\tClassification report for Artificial Neural Network\n\n",classification_report(test,pred,digits=6))
print()
print("Confusion Matrix for Artificial Neural Network")
print(confusion_matrix(test, pred))
print(test,pred)

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


		Classification report for Artificial Neural Network

               precision    recall  f1-score   support

           0   0.999049  0.996090  0.997567     76983
           1   0.999259  0.999108  0.999184     66152
           2   0.894737  0.944444  0.918919        18
           3   0.927059  0.989121  0.957085      3585
           4   0.000000  0.000000  0.000000         3

    accuracy                       0.997254    146741
   macro avg   0.764021  0.785753  0.774551    146741
weighted avg   0.997352  0.997254  0.997277    146741


Confusion Matrix for Artificial Neural Network
[[76682    33     0   268     0]
 [   47 66093     2    10     0]
 [    0     0    17     1     0]
 [   25    14     0  3546     0]
 [    1     2     0     0     0]]
[1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 

In [57]:
#Decision Tree Classifier with GridSearch
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
X = df.drop('category',axis=1)
y = df['category']
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=109)
dec = DecisionTreeClassifier()
params = {'max_depth':(1,2,3,10,100,1000)}
grid_search_dec = GridSearchCV(dec, params, cv=5, verbose=10, n_jobs=-1)
grid_search_dec.fit(X_train, Y_train)


Fitting 5 folds for each of 6 candidates, totalling 30 fits


GridSearchCV(cv=5, estimator=DecisionTreeClassifier(), n_jobs=-1,
             param_grid={'max_depth': (1, 2, 3, 10, 100, 1000)}, verbose=10)

In [58]:
grid_search_dec.best_params_

{'max_depth': 100}

In [59]:
y_pred = grid_search_dec.predict(X_test)

In [60]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt


print("\t\tClassification report for DecisionTree\n\n", classification_report(Y_test, y_pred, digits=6))
print()

print("Confusion Matrix for DecisionTree")
print(confusion_matrix(Y_test, y_pred))



		Classification report for DecisionTree

                 precision    recall  f1-score   support

          DDoS   0.999857  0.999935  0.999896     76983
           DoS   0.999864  0.999924  0.999894     66152
        Normal   1.000000  0.722222  0.838710        18
Reconnaissance   0.999441  0.998047  0.998744      3585
         Theft   1.000000  1.000000  1.000000         3

      accuracy                       0.999850    146741
     macro avg   0.999832  0.944026  0.967449    146741
  weighted avg   0.999850  0.999850  0.999847    146741


Confusion Matrix for DecisionTree
[[76978     5     0     0     0]
 [    4 66147     0     1     0]
 [    3     1    13     1     0]
 [    4     3     0  3578     0]
 [    0     0     0     0     3]]


In [61]:
#Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
randForest = RandomForestClassifier()
params = {'max_depth':(1,2,9,10),'n_estimators':(10,15,30)}
grid_search = GridSearchCV(randForest, params, cv=5, verbose=10)
grid_search.fit(X_train, Y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV 1/5; 1/12] START max_depth=1, n_estimators=10...............................
[CV 1/5; 1/12] END max_depth=1, n_estimators=10;, score=0.836 total time=   5.2s
[CV 2/5; 1/12] START max_depth=1, n_estimators=10...............................
[CV 2/5; 1/12] END max_depth=1, n_estimators=10;, score=0.802 total time=   4.0s
[CV 3/5; 1/12] START max_depth=1, n_estimators=10...............................
[CV 3/5; 1/12] END max_depth=1, n_estimators=10;, score=0.841 total time=   3.2s
[CV 4/5; 1/12] START max_depth=1, n_estimators=10...............................
[CV 4/5; 1/12] END max_depth=1, n_estimators=10;, score=0.801 total time=   3.1s
[CV 5/5; 1/12] START max_depth=1, n_estimators=10...............................
[CV 5/5; 1/12] END max_depth=1, n_estimators=10;, score=0.786 total time=   4.0s
[CV 1/5; 2/12] START max_depth=1, n_estimators=15...............................
[CV 1/5; 2/12] END max_depth=1, n_estimators=15;

GridSearchCV(cv=5, estimator=RandomForestClassifier(),
             param_grid={'max_depth': (1, 2, 9, 10),
                         'n_estimators': (10, 15, 30)},
             verbose=10)

In [62]:
y_pred = grid_search.predict(X_test)

In [63]:
grid_search.best_params_

{'max_depth': 10, 'n_estimators': 30}

In [64]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
print("\t\tClassification report for RandomForest\n\n",classification_report(Y_test,y_pred,digits=6))
print()
print()
print("Confusion Matrix for RandomForest")
print(confusion_matrix(Y_test, y_pred))


		Classification report for RandomForest

                 precision    recall  f1-score   support

          DDoS   0.999324  0.998636  0.998980     76983
           DoS   0.998414  0.999274  0.998844     66152
        Normal   0.937500  0.833333  0.882353        18
Reconnaissance   0.999442  0.998884  0.999163      3585
         Theft   1.000000  1.000000  1.000000         3

      accuracy                       0.998910    146741
     macro avg   0.986936  0.966026  0.975868    146741
  weighted avg   0.998909  0.998910  0.998909    146741



Confusion Matrix for RandomForest
[[76878   104     1     0     0]
 [   47 66104     0     1     0]
 [    2     0    15     1     0]
 [    3     1     0  3581     0]
 [    0     0     0     0     3]]
